<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 通过 Llama 3 与 Ollama 生成指令数据集


- 本 notebook 通过 Ollama 使用 80 亿参数 Llama 3 模型，采用 [Magpie: Alignment Data Synthesis from Scratch by Prompting Aligned LLMs with Nothing](https://arxiv.org/abs/2406.08464) 论文中提出的「技巧」生成合成数据集

- 生成的数据集是指令数据集，包含与 Alpaca 类似的 "instruction" 和 "output" 字段：


```python
{
    "instruction": "What is the atomic number of helium?",
    "output": "The atomic number of helium is 2.",
},
```

- 该代码不需要 GPU，可在笔记本电脑上运行（已在 M3 MacBook Air 上测试）

*注意：此处创建的指令数据集仅用于教育目的。用户有责任确保其使用符合 Meta AI Llama 3 相关许可协议的条款。*

In [ ]:
from importlib.metadata import version

pkgs = [
    "tqdm",    # 进度条
]

for p in pkgs:
    print(f"{p} 版本: {version(p)}")

## 安装 Ollama 并下载 Llama 3


- Ollama 是一款高效运行 LLM 的应用
- 它是 [llama.cpp](https://github.com/ggerganov/llama.cpp) 的封装，后者用纯 C/C++ 实现 LLM 以最大化效率
- 注意，它是用于 LLM 文本生成（推理）的工具，而非训练或微调 LLM
- 运行下方代码前，请访问 [https://ollama.com](https://ollama.com) 并按说明安装 ollama（例如点击「Download」按钮，下载适用于您操作系统的 ollama 应用）

- macOS 与 Windows 用户点击下载的 ollama 应用；若提示安装命令行，选择「是」
- Linux 用户可使用 ollama 官网提供的安装命令

- 通常，要从命令行使用 ollama，需先启动 ollama 应用，或在单独终端中运行 `ollama serve`

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/ollama-eval/ollama-serve.webp?1">


- 在 ollama 应用或 `ollama serve` 运行的情况下，在另一终端的命令行执行以下命令试用 80 亿参数 Llama 3 模型（该模型占用 4.7 GB 存储空间，首次执行此命令时会自动下载）

```bash
# 8B model
ollama run llama3
```


输出大致如下：

```
$ ollama run llama3
pulling manifest 
pulling 6a0746a1ec1a... 100% ▕████████████████▏ 4.7 GB                          
pulling 4fa551d4f938... 100% ▕████████████████▏  12 KB                          
pulling 8ab4849b038c... 100% ▕████████████████▏  254 B                          
pulling 577073ffcc6c... 100% ▕████████████████▏  110 B                          
pulling 3f8eb4da87fa... 100% ▕████████████████▏  485 B                          
verifying sha256 digest 
writing manifest 
removing any unused layers 
success 
```

- 注意，`llama3` 指经过指令微调的 80 亿参数 Llama 3 模型

- 若机器支持，也可将 `llama3` 替换为 `llama3:70b` 使用更大的 700 亿参数 Llama 3 模型

- 下载完成后，会出现命令行提示符，可与模型对话

- 可尝试类似 "What do llamas eat?" 的提示，应得到类似以下输出：

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered 
stomach and eat plants that are high in fiber. In the wild, llamas 
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall 
grasses, wheat, oats, and barley.
```

- 可使用 `/bye` 结束会话


## 使用 Ollama 的 REST API


- 现在，与模型交互的另一种方式是通过 Python 调用其 REST API，使用以下函数
- 运行本 notebook 后续单元格前，请确保 ollama 仍在运行，方式同上：
  - 在终端中运行 `ollama serve`
  - 或启动 ollama 应用
- 接下来，运行以下代码单元格查询模型

- 首先用简单示例测试 API，确保其按预期工作：


In [ ]:
import json
import requests

def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat", role="user"):
    # 将数据 payload 创建为字典
    data = {
        "model": model,
        "seed": 123,        # 用于确定性响应
        "temperature": 1.,   # 用于确定性响应
        "top_p": 1,         
        "messages": [
            {"role": role, "content": prompt}
        ]
    }

    # 发送 POST 请求
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data

result = query_model("What do Llamas eat?")
print(result)

In [ ]:
result = query_model("What do Llamas eat?")
print(result)

## 提取指令


- 现在，我们使用论文中提出的「技巧」：提供空提示模板 `"<|begin_of_text|><|start_header_id|>user<|end_header_id|>"`，这将使经过指令微调的 Llama 3 模型生成一条指令

In [ ]:
def extract_instruction(text):
    for content in text.split("\n"):
        if content:
            return content.strip()

In [ ]:
query = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>"

result = query_model(query, role="assistant")
instruction = extract_instruction(result)
print(instruction)

- 如上所示，令人惊讶的是，模型确实生成了指令


## 生成回复


- 下一步是创建对应回复，只需将指令作为输入传入即可


In [ ]:
response = query_model(instruction, role="user")
print(response)

## 生成数据集


- 我们可以将此方法扩展到任意数量的数据样本（您可能希望应用一些可选的长度或质量过滤，例如使用另一个 LLM 对生成数据进行评分）
- 下方，我们生成 5 对合成指令-回复，在 M3 MacBook Air 上大约需要 3 分钟
- （要生成适合指令微调的数据集，我们需要将其增加到至少 1k 到 50k，并可能在 GPU 上运行以更快生成样本）

**提示**

- 将 `model="llama3"` 改为 `model="llama3:70b"` 可生成更高质量的回复，但这需要更多计算资源

In [ ]:
from tqdm import tqdm

dataset_size = 5
dataset = []

for i in tqdm(range(dataset_size)):

    result = query_model(query, role="assistant")
    instruction = extract_instruction(result)
    response = query_model(instruction, role="user")
    entry = {
        "instruction": instruction,
        "output": response
    }
    dataset.append(entry)

In [ ]:
with open("instruction-data-llama3-7b.json", "w") as file:
    json.dump(dataset, file, indent=4)

In [ ]:
!cat instruction-data-llama3-7b.json